# SAIFEN — 03 · KDE + Heatmap

Geração dos artefatos finais que o frontend consome.

**Output:**

1. `output/heatmaps/heatmap_points.json`  — Leaflet.heat
   (`[[lat, lng, weight], ...]`)
2. `output/heatmaps/heatmap_grid.geojson` — Mapbox GL JS / PostGIS
   (FeatureCollection de Polygons com `properties.density`)
3. `output/heatmaps/crimes.geojson`       — Pontos individuais (debug)

A bandwidth padrão usa a regra de Scott (`scott`). Para zoom mais
"explodido" (manchas mais largas), passe um float maior, ex. `0.003`.

In [ ]:
import sys
from pathlib import Path

NB_DIR = Path.cwd()
PROJECT_ROOT = NB_DIR.parent if NB_DIR.name == "notebooks" else NB_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from saifen_pipeline import config, kde, exporter

## 1. Carregar dataset limpo

Lê o parquet gerado no notebook 02. Se não existir, rode `02_limpeza.ipynb`
antes (ou `python scripts/process_xlsx.py`).

In [ ]:
parquet_path = config.PROCESSED_DIR / "celulares_clean.parquet"
df = pd.read_parquet(parquet_path)
print(f"Pontos: {len(df):,}")
df.head()

## 2. KDE em grade (heatmap analítico)

`kde.grid_density` constrói o gaussian_kde 2D e avalia em N×N células.
Use `grid_size=200` para web (rápido) ou `400` para análises de detalhe.

In [ ]:
grid = kde.grid_density(
    df,
    bbox=config.SP_BBOX,
    grid_size=200,
    bandwidth="scott",
    sample_max=10_000,
)
print(f"Grid: {grid.zi.shape}, bandwidth efetiva (factor): {grid.bandwidth:.4f}")
print(f"Bbox: {grid.bbox}")
print(f"Densidade — min={grid.zi.min():.4f} | max={grid.zi.max():.4f} | mean={grid.zi.mean():.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 8))
extent = [grid.bbox[0], grid.bbox[2], grid.bbox[1], grid.bbox[3]]
im = ax.imshow(
    grid.zi,
    origin="lower",
    extent=extent,
    cmap="inferno",
    aspect="equal",
)
ax.set_facecolor("#000")
fig.patch.set_facecolor("#000")
ax.tick_params(colors="#00FF9F")
for spine in ax.spines.values():
    spine.set_color("#00FF9F")
ax.set_title("KDE — densidade normalizada", color="#00FF9F")
ax.set_xlabel("Longitude", color="#00FF9F")
ax.set_ylabel("Latitude", color="#00FF9F")
fig.colorbar(im, ax=ax, label="densidade")
plt.show()

## 3. Pontos pesados para Leaflet.heat

Mantém compatibilidade com o frontend atual (`crimeHeatData` em
`js/data/MockData.js`).

In [ ]:
points = kde.point_heatmap(df, sample_max=5_000, bandwidth="scott")
print(f"Pontos gerados: {len(points)}")
print("Exemplo:", points[0])

## 4. Exportar tudo para `output/heatmaps/`

Esses arquivos são versionados (entram em git) — são os artefatos
estáticos que o frontend / GitHub Pages serve.

In [ ]:
paths = {
    "heatmap_points":  exporter.write_heatmap_points(points),
    "heatmap_grid":    exporter.write_heatmap_grid(grid, min_density=0.05),
    "crimes_geojson":  exporter.write_crimes_geojson(df.sample(min(5_000, len(df)), random_state=0)),
}

for name, p in paths.items():
    size_kb = p.stat().st_size / 1024
    print(f"{name:20s} → {p.relative_to(config.ROOT_DIR)}  ({size_kb:,.1f} KB)")

## 5. Heatmaps fatiados por tipo / período (opcional)

Para o filtro do frontend (`furto/roubo` × `manhã/tarde/noite/madrugada`),
gere um arquivo por combinação. Útil para Mapbox `setData` ao trocar
camada sem reprocessar no browser.

In [ ]:
for ctype in ["furto", "roubo"]:
    subset = df[df["crime_type"] == ctype]
    if len(subset) < 100:
        continue
    pts = kde.point_heatmap(subset, sample_max=3_000, bandwidth="scott")
    out = exporter.write_heatmap_points(
        pts,
        path=config.HEATMAP_DIR / f"heatmap_points__{ctype}.json",
        crime_type=ctype,
    )
    print(f"{ctype:8s} ({len(subset):>6,} pontos) → {out.name}")